# Day 2 Lab - Forecasting using Machine Learning
### AIOps Fundamentals Training | Microland

---

## What This Notebook Does

This notebook generates continuous hourly data points over 10 weeks (1680 records per server node). Every Friday between 20:00 and 23:00 UTC, the network throughput dynamically scales up above 80% utilization, while maintaining a normal weekend evening bump and a low weekday baseline

Once uploaded to Elastic Cloud, you will create an ML Anomaly Detection job in Kibana and use Forecast option to predict future value using Elastic Machine Learning.

---



---

## Before You Start
1. you have a running Elastic Cloud deployment
2. You have your **Cloud ID** and **API Key** ready
3. Running in **Google Colab**

> **No Python knowledge required.** Run each cell in order using the Play button or Shift+Enter.


## Step 1 - Install Required Libraries


In [14]:
!pip install elasticsearch --quiet
print('Libraries installed.')


Libraries installed.


## Step 2 - Configuration

Enter your Elastic Cloud credentials below.
You can also adjust the simulation parameters if you want to experiment.


In [15]:
# -----------------------------------------------------------
# ELASTIC CLOUD CREDENTIALS - fill in your own values
# -----------------------------------------------------------
CLOUD_ID   = 'Your Elastic Cloud ID'
API_KEY    = 'Your Elasic API'
print("Configuration Done: Cloud ID and API updated")


Configuration Done: Cloud ID and API updated


## Step 3 - Generate the Network Utilization Data over 10 Weeks Time



In [ ]:
import datetime
import random
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

# --- CONFIGURATION ---

INDEX_NAME = "network-bandwidth-forecasting-log"

print("Connecting to Elastic Cloud...")
es = Elasticsearch(cloud_id=CLOUD_ID, api_key=API_KEY)

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f"Resetting existing index: {INDEX_NAME}")

# Updated Timeline: 10 Weeks of History (approx 1,680 hourly data points)
weeks_of_history = 10
end_time = datetime.datetime.utcnow()
start_time = end_time - datetime.timedelta(weeks=weeks_of_history)

print(f"Generating hourly data from {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}...")

# Pre-determine 4 completely random hours across the 10 weeks to inject true anomalies
total_hours = weeks_of_history * 7 * 24
random_anomaly_hours = random.sample(range(100, total_hours - 100), 4)

actions = []
current_cursor = start_time
max_link_speed_mbps = 1000.0

for h in range(total_hours):
    weekday = current_cursor.weekday()
    hour = current_cursor.hour

    # 1. Normal low baseline weekday usage (10% to 25%)
    base_util_pct = random.uniform(10.0, 25.0)
    log_level = "INFO"
    message = "Streaming gateway interface performance nominal."
    status = "Healthy"

    # 2. General weekend uplift behavior (Saturday & Sunday evening traffic)
    if weekday in [5, 6] and (17 <= hour <= 23):
        base_util_pct = random.uniform(45.0, 65.0)

    # 3. Known Cyclical Event: Every Friday Night between 8:00 PM and 11:00 PM (20:00 to 23:00)
    is_friday_peak = (weekday == 4) and (20 <= hour <= 23)
    if is_friday_peak:
        base_util_pct = random.uniform(82.0, 96.5)
        message = "High outbound socket utilization due to concentrated regional multimedia streaming demands."

    # 4. Inject Malicious/Random Telemetry Spikes (True Anomalies)
    # This happens on a random Tuesday/Wednesday etc., outside the regular weekend patterns
    if h in random_anomaly_hours:
        base_util_pct = random.uniform(92.0, 99.1)  # Massive unexpected spike
        log_level = "CRITICAL"
        status = "Critical"
        message = "ALERT: Unexpected outbound saturation anomaly detected. Non-standard traffic signature."
        print(f"   [Simulation Log] Injected random anomaly at: {current_cursor.strftime('%Y-%m-%d %H:%M')}")

    # Calculate absolute bandwidth values
    network_in = (base_util_pct / 100.0) * max_link_speed_mbps
    network_out = network_in * random.uniform(1.1, 1.3)

    doc = {
        "_index": INDEX_NAME,
        "_source": {
            "@timestamp": current_cursor.isoformat() + "Z",
            "service.name": "microland-cinema-cdn",
            "system.network.utilization_pct": round(base_util_pct, 2),
            "system.network.in_mbps": round(network_in, 2),
            "system.network.out_mbps": round(network_out, 2),
            "system.status": status,
            "log.level": log_level,
            "message": message
        }
    }
    actions.append(doc)
    current_cursor += datetime.timedelta(hours=1)

print(f"Uploading {len(actions)} historical metric records to Elastic...")
bulk(es, actions)
es.indices.refresh(index=INDEX_NAME)
print("Data ingestion complete!")

## Step 9 - Create the ML Anomaly Detection and Forecasting
---
### 9a - Create a dataview named: `network-bandwidth-forecasting`
---

### 9b - Create the Anomaly Detection Job

1. In Kibana, go to: **Machine Learning > Anomaly Detection > Create Job**
2. Select Dataview: `network-bandwidth-forecasting`
3. Choose job type: **Single Metric**
4. Configure the job:

| Setting | Value |
|---------|-------|
| Function | `mean` |
| Field | `system.network.utilization_pct` |
| Bucket span | `1h` |
| Job ID | `network-bandwidth-forecasting` |

5. Set the time range to **Full** (to cover all 10 weeks of data)
6. Click **Create Job** and wait for it to complete

---

### 9c - Analyse the Results in Anomaly Explorer

1. Go to **Machine Learning > Anomaly Explorer**
2. Select your job: `network-bandwidth-forecasting`
3. Look for the swimlane heatmap - you should see a red/orange block
4. Click on the anomaly block and note:
   - The **anomaly score**
   - The **actual value**
   - The **typical value** (what the ML expected to see at that time)

---
### 9d - Generate 14 or 21- Day Forecast

Once the ML job finishes processing the historical data, look at the top toolbar of the Single Metric Viewer and click the Forecast button.

In the configuration popup, set the duration parameters:

Duration: 14d (or 21d)

Click Run.


---


### Explanation

Introducing random, unpredictable spikes during the week creates a realistic environment. This allows you to show students the two distinct capabilities of Elastic ML simultaneously:

**Anomaly Detection:** The random, unexpected mid-week spikes will be flagged as anomalies (high anomaly scores) because they violate the baseline.

**Seasonal Learning & Forecasting:** The consistent Friday night streaming surge will still be learned as normal behavior, and the 14-to-21 day forecast will accurately project those weekend peaks into the future while ignoring the random mid-week noise.

---


